In [0]:
display(spark.table("default.events_delta").filter("event_type = 'purchase'").limit(10))

In [0]:
spark.sql("""
  CREATE TABLE silver_events_part
  USING DELTA
  PARTITIONED BY (event_date, event_type)
  AS SELECT *, date(event_time) AS event_date FROM events_delta
""")

In [0]:
# Optimize
spark.sql("OPTIMIZE silver_events_part ZORDER BY (user_id, product_id)")

# Benchmark
import time
start = time.time()
spark.sql("SELECT * FROM silver_events_part WHERE user_id=12345").count()
print(f"Time: {time.time()-start:.2f}s")

# Cache for iterative queries
#cached = spark.table("silver_events_part").cache()
#cached.count()  # Materialize